In [11]:
#@title Cell 1: Install required libraries
!pip install -q google-genai matplotlib pandas numpy

In [12]:
#@title Cell 2: Imports, API configuration, and global constants
import time
import random
import re
import json
import os
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- API Configuration (New Library) ---
from google import genai
from google.colab import userdata

API_KEY = userdata.get('project')
client = genai.Client(api_key=API_KEY)

# --- Model names ---
LARGE_MODEL = "gemma-3-27b-it"   # For candidates & moderator
SMALL_MODEL = "gemma-3-12b-it"   # For voters (higher rate limit)

# --- Debate topics ---
DEBATE_TOPICS = [
    "China surpassing America as the dominant global power",
    "The US Healthcare system",
    "Gun ownership and carry permits in America"
]

# --- Rate Limit Settings ---
BATCH_SIZE = 5          # Number of voters per batch
BATCH_DELAY = 65        # Seconds to wait between batches
CTX_LIMIT = 1500        # Max context chars to save tokens

# --- Cache Settings ---
DEBATE_CACHE_FILE = "debate_transcript.json"
VOTERS_CACHE_FILE = "voters_cache.json"

print("✅ Setup complete.")
print(f"📊 Models: {LARGE_MODEL} (large), {SMALL_MODEL} (small)")
print(f"⏱️ Rate limit settings: {BATCH_SIZE} voters/batch, {BATCH_DELAY}s delay")

✅ Setup complete.
📊 Models: gemma-3-27b-it (large), gemma-3-12b-it (small)
⏱️ Rate limit settings: 5 voters/batch, 65s delay


In [13]:
#@title Cell 3: Core LLM utilities with robust error handling

import time
import random

def call_model(model_name, prompt, max_tokens=512, max_retries=8, initial_wait=20):
    """
    Send a prompt to a Gemma model with robust error handling.
    Handles: Rate Limit (429), Server Unavailable (503), and other errors.
    """
    wait = initial_wait

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config={
                    "max_output_tokens": max_tokens,
                    "temperature": 0.8
                }
            )
            return response.text.strip()

        except Exception as e:
            error_str = str(e)
            error_code = ""

            # Extract error code
            if "429" in error_str:
                error_code = "429 (Rate Limit)"
            elif "503" in error_str:
                error_code = "503 (Server Busy)"
            elif "500" in error_str:
                error_code = "500 (Server Error)"
            elif "ResourceExhausted" in error_str:
                error_code = "ResourceExhausted"
            elif "UNAVAILABLE" in error_str:
                error_code = "UNAVAILABLE"

            if error_code:
                # Add jitter to avoid thundering herd
                jitter = random.uniform(0.5, 1.5)
                actual_wait = wait * jitter

                print(f"  ⚠️ {error_code} - Attempt {attempt+1}/{max_retries}")
                print(f"  ⏳ Waiting {actual_wait:.0f}s (base: {wait}s)...")

                time.sleep(actual_wait)

                # Exponential backoff with cap
                wait = min(wait * 2, 300)  # Max 5 minutes
            else:
                # Unknown error - still retry but log it
                print(f"  ❌ Unknown error: {error_str[:150]}")
                print(f"  🔄 Retrying in {wait}s...")
                time.sleep(wait)
                wait = min(wait * 1.5, 180)

    # All retries exhausted
    raise RuntimeError(f"Max retries ({max_retries}) exceeded. Last error: {error_str[:200]}")


def safe_call_model(model_name, prompt, max_tokens=512, fallback_response="[No response]"):
    """
    Wrapper that never raises - returns fallback on failure.
    Useful for non-critical calls.
    """
    try:
        return call_model(model_name, prompt, max_tokens)
    except Exception as e:
        print(f"  ⚠️ Using fallback response due to: {str(e)[:100]}")
        return fallback_response


def add_delay(seconds, message=""):
    """Add a delay with progress indication."""
    if message:
        print(f"  ⏳ {message}")

    # Show progress for long waits
    if seconds > 30:
        for remaining in range(seconds, 0, -15):
            time.sleep(min(15, remaining))
            if remaining > 15:
                print(f"      ⏳ {remaining}s remaining...")
    else:
        time.sleep(seconds)


print("✅ Core utilities defined with robust error handling.")
print("   • call_model() - With retry logic for 429/503 errors")
print("   • safe_call_model() - Never raises, uses fallback")
print("   • add_delay() - Progress-aware delay")

✅ Core utilities defined with robust error handling.
   • call_model() - With retry logic for 429/503 errors
   • safe_call_model() - Never raises, uses fallback
   • add_delay() - Progress-aware delay


In [14]:
#@title Cell 4: Phase 1 - Candidate Agents

def build_persona(name, party, traits):
    """Build a persona string for a candidate based on name, party, and traits."""
    trait_lines = "\n".join(f"  - {k}: {v}/10" for k, v in traits.items())
    return (
        f"You are {name}, a {party} presidential candidate.\n"
        f"Personality traits (1-10 scale):\n{trait_lines}\n"
        f"Always respond fully in character. Let these traits shape your tone and answers.\n"
        f"Be consistent with your political party's typical positions."
    )


def create_candidate(name, party, traits):
    """Create and return a candidate agent dict."""
    return {
        "name": name,
        "party": party,
        "traits": traits,
        "persona": build_persona(name, party, traits)
    }


def candidate_answer(candidate, context, question):
    """Generate a candidate's in-character answer to a debate question."""
    prompt = (
        f"{candidate['persona']}\n\n"
        f"Debate context so far:\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Give a detailed, in-character response (4-6 sentences). "
        f"Stay true to your personality traits and political positions."
    )
    return call_model(LARGE_MODEL, prompt, max_tokens=400)


def evaluate_candidate(candidate, test_q):
    """Test a candidate agent with a controversial question."""
    prompt = (
        f"{candidate['persona']}\n\n"
        f"Answer this question honestly (your honesty={candidate['traits'].get('honesty',5)}/10):\n"
        f"{test_q}"
    )
    answer = call_model(LARGE_MODEL, prompt, max_tokens=300)
    print(f"\n[{candidate['name']}] → {answer[:500]}...")
    return answer


# ══════════════════════════════════════════════════════════════
# Define the two candidates
# ══════════════════════════════════════════════════════════════

candidate_A = create_candidate(
    name   = "Donald Trump",
    party  = "Republican",
    traits = {
        "aggression": 9,
        "charisma": 8,
        "honesty": 4,
        "empathy": 3,
        "intelligence": 7
    }
)

candidate_B = create_candidate(
    name   = "Kamala Harris",
    party  = "Democrat",
    traits = {
        "aggression": 5,
        "charisma": 7,
        "honesty": 7,
        "empathy": 8,
        "intelligence": 8
    }
)

CANDIDATES = [candidate_A, candidate_B]

print("✅ Phase 1 — Candidate agents created:")
for c in CANDIDATES:
    print(f"   • {c['name']} ({c['party']})")
    print(f"     Traits: {c['traits']}")

# Quick evaluation test
print("\n" + "="*60)
print(" PHASE 1 EVALUATION TEST")
print("="*60)

TEST_QUESTION = "Would you ever consider changing your stance on gun control to gain more votes?"
for c in CANDIDATES:
    evaluate_candidate(c, TEST_QUESTION)
    time.sleep(5)  # Small delay between candidates

✅ Phase 1 — Candidate agents created:
   • Donald Trump (Republican)
     Traits: {'aggression': 9, 'charisma': 8, 'honesty': 4, 'empathy': 3, 'intelligence': 7}
   • Kamala Harris (Democrat)
     Traits: {'aggression': 5, 'charisma': 7, 'honesty': 7, 'empathy': 8, 'intelligence': 8}

 PHASE 1 EVALUATION TEST

[Donald Trump] → Look, let me tell you, this is a very, very important question. A *tremendous* question, frankly. And the fake news media, they'll try to twist my words, they always do. Sad!

But here's the deal. I am a staunch defender of the Second Amendment. The BEST defender. Believe me. Our country was founded on the right to bear arms, and frankly, we need that right now more than ever. We have bad people out there, very bad people, and good people need to be able to defend themselves. It's simple.

Now, ...

[Kamala Harris] → (Adjusts my jacket, offers a warm but firm smile)

That's a very direct question, and I appreciate you asking it. Let me be clear: the safety of our

In [15]:
#@title Cell 5: Phase 2 - Moderator Agent & Debate Simulation

MODERATOR_PERSONA = (
    "You are a neutral, professional, and sharp presidential debate moderator. "
    "Ask challenging follow-up questions and briefly critique candidate answers. "
    "Be fair to both candidates but don't let them dodge questions. "
    "Always refer to candidates by their correct names: Donald Trump and Kamala Harris."
)


def moderator_intro(candidates):
    """Generate the moderator's opening introduction for the debate."""
    names = " and ".join(f"{c['name']} ({c['party']})" for c in candidates)
    prompt = (
        f"{MODERATOR_PERSONA}\n\n"
        f"Introduce tonight's presidential debate between {names}. "
        f"Be engaging and professional (3-4 sentences). "
        f"Mention the format: 3 topics with 5 questions each."
    )
    return call_model(LARGE_MODEL, prompt, max_tokens=250)


def moderator_question(topic, context, q_num):
    """Generate a moderator question."""
    prompt = (
        f"{MODERATOR_PERSONA}\n\n"
        f"Topic: {topic}\n"
        f"Context (recent):\n{context[-CTX_LIMIT:]}\n\n"
        f"Ask debate question #{q_num} on this topic. "
        f"Make it sharp, direct, and challenging (1-2 sentences)."
    )
    return call_model(LARGE_MODEL, prompt, max_tokens=150)


def moderator_critique(context):
    """Generate a brief moderator critique."""
    prompt = (
        f"{MODERATOR_PERSONA}\n\n"
        f"Context (recent):\n{context[-CTX_LIMIT:]}\n\n"
        f"Briefly critique the last two answers from Donald Trump and Kamala Harris. "
        f"Be fair but pointed (2 sentences max)."
    )
    return call_model(LARGE_MODEL, prompt, max_tokens=120)


def run_debate(candidates, topics, questions_per_topic=5):
    """
    Run the full debate simulation.
    Returns the complete transcript string and final context.
    """
    transcript = []

    # Opening introduction
    print("\n🎤 Generating moderator introduction...")
    intro = moderator_intro(candidates)
    transcript.append(f"[MODERATOR]: {intro}\n")
    print(f"\n{'='*70}")
    print(f"[MODERATOR INTRO]: {intro}")
    context = intro

    for topic_idx, topic in enumerate(topics):
        header = f"\n{'='*70}\n📌 TOPIC {topic_idx+1}: {topic}\n{'='*70}"
        transcript.append(header)
        print(header)

        order = candidates.copy()
        random.shuffle(order)  # Random speaking order per topic

        for q_num in range(1, questions_per_topic + 1):
            print(f"\n--- Question {q_num}/{questions_per_topic} ---")

            # Moderator asks a question
            question = moderator_question(topic, context, q_num)
            transcript.append(f"\n[MOD Q{q_num}]: {question}")
            print(f"[MOD Q{q_num}]: {question}")
            context += f"\nModerator: {question}"

            time.sleep(3)  # Small delay

            # Each candidate answers
            for candidate in order:
                print(f"  💬 {candidate['name']} answering...")
                answer = candidate_answer(candidate, context[-CTX_LIMIT:], question)
                line = f"[{candidate['name'].upper()}]: {answer}"
                transcript.append(line)
                print(f"[{candidate['name'].upper()}]: {answer[:300]}...")
                context += f"\n{candidate['name']}: {answer}"
                time.sleep(3)  # Delay between candidates

            # Moderator critiques
            critique = moderator_critique(context)
            transcript.append(f"[MOD CRITIQUE]: {critique}")
            print(f"[MOD CRITIQUE]: {critique}")
            context += f"\nModerator critique: {critique}"

            order = order[::-1]  # Alternate speaking order

            # Delay between questions to avoid rate limit
            if q_num < questions_per_topic:
                time.sleep(5)

        # Delay between topics
        if topic_idx < len(topics) - 1:
            print(f"\n⏳ Waiting 30s before next topic...")
            time.sleep(30)

    full_transcript = "\n".join(transcript)
    return full_transcript, context


def get_or_run_debate(candidates, topics, questions_per_topic, use_cache=True):
    """
    Get debate from cache if available, otherwise run new debate.
    """
    if use_cache and os.path.exists(DEBATE_CACHE_FILE):
        print("📂 Loading debate from cache...")
        with open(DEBATE_CACHE_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            print(f"   Loaded transcript: {len(data['transcript'])} chars")
            return data['transcript'], data['context']

    # Run new debate
    print("🎤 Running new debate (this may take a while)...")
    transcript, context = run_debate(candidates, topics, questions_per_topic)

    # Save to cache
    with open(DEBATE_CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump({'transcript': transcript, 'context': context}, f, ensure_ascii=False)
    print(f"💾 Debate saved to cache: {DEBATE_CACHE_FILE}")

    return transcript, context


print("✅ Phase 2 functions defined.")
print("   • moderator_intro()")
print("   • moderator_question()")
print("   • moderator_critique()")
print("   • run_debate()")
print("   • get_or_run_debate() [with caching]")

✅ Phase 2 functions defined.
   • moderator_intro()
   • moderator_question()
   • moderator_critique()
   • run_debate()
   • get_or_run_debate() [with caching]


In [16]:
#@title Cell 6: Phase 3 - Voter Agents with Resume Capability

import json
import os

# ══════════════════════════════════════════════════════════════
# VALUE SETS (same as before)
# ══════════════════════════════════════════════════════════════

def generate_value_sets(n=200):
    """Generate n diverse value sets for voters."""

    china_positions = [
        "America must counter China aggressively — militarily if needed.",
        "We need strong economic sanctions and military readiness against China.",
        "Trade competition, not military rivalry, is the right approach to China.",
        "Multilateral diplomacy with allies is how we manage China's rise.",
        "We should seek peaceful coexistence and cooperation with China.",
        "China's rise is inevitable; we should focus on domestic issues.",
        "Economic interdependence with China benefits both nations.",
        "We need balanced approach: firm on security, open on trade.",
        "Technology competition is the key battleground with China.",
        "Human rights must be central to our China policy."
    ]

    healthcare_positions = [
        "Healthcare is a right; the government must ensure universal coverage.",
        "Medicare for All is the only moral healthcare system.",
        "A strong public option alongside private insurance is ideal.",
        "Expand ACA but keep private insurance as the backbone.",
        "A hybrid public-private healthcare system offers the best balance.",
        "Free-market competition drives better outcomes in healthcare.",
        "Government should only provide catastrophic coverage.",
        "Healthcare is a personal responsibility, not government's job.",
        "State-level solutions are better than federal healthcare programs.",
        "Tax incentives for health savings accounts are the answer."
    ]

    gun_positions = [
        "I support strict gun control; assault weapons have no place in civilian hands.",
        "Universal background checks and red flag laws are essential.",
        "Ban assault weapons but respect hunting and self-defense rights.",
        "Reasonable firearm regulations are compatible with the Second Amendment.",
        "Focus on mental health, not gun restrictions.",
        "Gun ownership is a constitutional right with minimal restrictions.",
        "The Second Amendment is absolute; citizens can carry arms freely.",
        "Armed citizens are the best defense against tyranny and crime.",
        "Concealed carry should be legal nationwide.",
        "Any gun control is a slippery slope to confiscation."
    ]

    value_sets = []
    for i in range(n):
        china_idx = (i * 7) % len(china_positions)
        health_idx = (i * 11) % len(healthcare_positions)
        gun_idx = (i * 13) % len(gun_positions)

        if random.random() < 0.3:
            china_idx = random.randint(0, len(china_positions)-1)
        if random.random() < 0.3:
            health_idx = random.randint(0, len(healthcare_positions)-1)
        if random.random() < 0.3:
            gun_idx = random.randint(0, len(gun_positions)-1)

        value_sets.append({
            "china": china_positions[china_idx],
            "healthcare": healthcare_positions[health_idx],
            "guns": gun_positions[gun_idx]
        })

    return value_sets


VALUE_SETS = generate_value_sets(200)
LEVEL_MAP = {"low": (1, 3), "medium": (4, 7), "high": (8, 10)}


def create_voter(voter_id, traits, values):
    """Create and return a single voter agent dict."""
    return {
        "id": voter_id,
        "traits": traits,
        "values": values,
        "vote": None,
        "explanation": None
    }


def generate_voters(n, wisdom_dist=None, fear_dist=None):
    """Generate n voter agents with random traits."""
    wisdom_dist = wisdom_dist or {"low": 0.33, "medium": 0.34, "high": 0.33}
    fear_dist = fear_dist or {"low": 0.33, "medium": 0.34, "high": 0.33}

    def sample_level(dist):
        return random.choices(
            ["low", "medium", "high"],
            weights=[dist["low"], dist["medium"], dist["high"]]
        )[0]

    voters = []
    for i in range(n):
        w_level = sample_level(wisdom_dist)
        f_level = sample_level(fear_dist)

        w = random.randint(*LEVEL_MAP[w_level])
        f = random.randint(*LEVEL_MAP[f_level])

        traits = {
            "wisdom": w,
            "fear": f,
            "anger": random.randint(1, 10),
            "openness": random.randint(1, 10),
            "trust": random.randint(1, 10)
        }

        value_set = VALUE_SETS[i % len(VALUE_SETS)]
        voters.append(create_voter(i + 1, traits, value_set))

    return voters


def voter_vote(voter, transcript_summary, candidates):
    """
    Simulate a voter casting a vote.
    Returns tuple: (chosen candidate name, explanation)
    """
    cand_names = [c["name"] for c in candidates]
    values_str = "\n".join(f"- {k}: {v}" for k, v in voter["values"].items())
    traits_str = ", ".join(f"{k}={v}/10" for k, v in voter["traits"].items())
    names_str = " or ".join(cand_names)

    prompt = (
        f"You are an American voter with these personality traits: {traits_str}.\n"
        f"Your political values and beliefs:\n{values_str}\n\n"
        f"You watched this presidential debate:\n{transcript_summary[:2500]}\n\n"
        f"Based on your personality and values, who do you vote for: {names_str}?\n\n"
        f"First, explain your reasoning in 2-3 sentences based on your traits and values.\n"
        f"Then on a new line, write ONLY: VOTE: [candidate full name]\n"
        f"Example: VOTE: Donald Trump"
    )

    raw = call_model(SMALL_MODEL, prompt, max_tokens=200)

    # Extract vote
    vote = None
    for name in cand_names:
        if name.lower() in raw.lower():
            vote = name
            break

    if vote is None:
        vote = random.choice(cand_names)

    explanation = raw.split("VOTE:")[0].strip() if "VOTE:" in raw else raw[:200]

    return vote, explanation


# ══════════════════════════════════════════════════════════════
# CHECKPOINT / RESUME SYSTEM
# ══════════════════════════════════════════════════════════════

CHECKPOINT_FILE = "election_checkpoint.json"


def save_checkpoint(voters, results, last_voter_idx):
    """Save current progress to checkpoint file."""
    checkpoint = {
        "last_voter_idx": last_voter_idx,
        "results": results,
        "voters": [
            {
                "id": v["id"],
                "traits": v["traits"],
                "values": v["values"],
                "vote": v.get("vote"),
                "explanation": v.get("explanation")
            }
            for v in voters
        ]
    }

    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)

    print(f"  💾 Checkpoint saved at voter {last_voter_idx + 1}")


def load_checkpoint():
    """Load progress from checkpoint file if exists."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None


def clear_checkpoint():
    """Remove checkpoint file after successful completion."""
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("  🗑️ Checkpoint cleared")


def run_election_with_resume(voters, transcript, candidates,
                              batch_size=5, delay_sec=65,
                              checkpoint_every=5):
    """
    Run election with automatic checkpoint/resume capability.
    If interrupted, can resume from last checkpoint.
    """

    # Check for existing checkpoint
    checkpoint = load_checkpoint()

    if checkpoint:
        print("="*60)
        print(" 📂 CHECKPOINT FOUND - RESUMING")
        print("="*60)

        start_idx = checkpoint["last_voter_idx"] + 1
        results = checkpoint["results"]

        # Restore voter states
        for i, v_data in enumerate(checkpoint["voters"]):
            if v_data.get("vote"):
                voters[i]["vote"] = v_data["vote"]
                voters[i]["explanation"] = v_data.get("explanation")

        completed = sum(1 for v in voters if v.get("vote"))
        print(f"   Resuming from voter {start_idx + 1}")
        print(f"   Already completed: {completed}/{len(voters)}")
        print(f"   Current results: {results}")
        print()
    else:
        start_idx = 0
        results = {c["name"]: 0 for c in candidates}

        # Count any pre-existing votes (shouldn't happen but just in case)
        for v in voters:
            if v.get("vote"):
                results[v["vote"]] += 1

    total_voters = len(voters)
    total_batches = (total_voters - start_idx - 1) // batch_size + 1

    print(f"\n🗳️ {'Resuming' if checkpoint else 'Starting'} election")
    print(f"   Voters remaining: {total_voters - start_idx}")
    print(f"   Batch size: {batch_size}")
    print(f"   Estimated time: ~{total_batches * delay_sec // 60} minutes\n")

    current_batch_start = start_idx

    for i in range(start_idx, total_voters):
        voter = voters[i]

        # Skip if already voted (from checkpoint)
        if voter.get("vote"):
            continue

        # Batch header
        if (i - start_idx) % batch_size == 0:
            batch_num = (i - start_idx) // batch_size + 1
            print(f"\n📦 Batch {batch_num}/{total_batches}")
            print("-" * 40)
            current_batch_start = i

        # Vote
        try:
            traits_short = f"W:{voter['traits']['wisdom']} F:{voter['traits']['fear']}"

            choice, explanation = voter_vote(voter, transcript, candidates)
            voter["vote"] = choice
            voter["explanation"] = explanation
            results[choice] += 1

            print(f"  Voter {voter['id']:>3} [{traits_short}] → {choice}")

            # Small delay between voters
            time.sleep(2)

        except Exception as e:
            print(f"\n  ❌ Error at voter {voter['id']}: {str(e)[:100]}")
            print(f"  💾 Saving checkpoint and pausing...")

            # Save checkpoint before the failed voter
            save_checkpoint(voters, results, i - 1)

            # Wait longer before retry
            print(f"  ⏳ Waiting 120s before retry...")
            time.sleep(120)

            # Retry this voter
            try:
                choice, explanation = voter_vote(voter, transcript, candidates)
                voter["vote"] = choice
                voter["explanation"] = explanation
                results[choice] += 1
                print(f"  ✅ Retry successful: Voter {voter['id']} → {choice}")
            except Exception as e2:
                print(f"  ❌ Retry failed: {str(e2)[:100]}")
                print(f"  ⚠️ Skipping voter {voter['id']} - Run again to retry")
                save_checkpoint(voters, results, i)
                raise  # Re-raise to stop execution

        # Checkpoint every N voters
        if (i + 1) % checkpoint_every == 0:
            save_checkpoint(voters, results, i)

        # Batch delay
        if (i - start_idx + 1) % batch_size == 0 and i < total_voters - 1:
            # Progress update
            votes_so_far = sum(results.values())
            print(f"\n  📊 Progress: {votes_so_far}/{total_voters} votes")
            for name, count in results.items():
                pct = count / votes_so_far * 100 if votes_so_far > 0 else 0
                bar = "█" * int(pct / 5)
                print(f"      {name}: {count} ({pct:.1f}%) {bar}")

            print(f"\n  ⏳ Rate limit pause: {delay_sec}s...")
            time.sleep(delay_sec)

    # Election complete - clear checkpoint
    clear_checkpoint()

    # Final results
    print("\n" + "="*60)
    print(" ✅ ELECTION COMPLETE")
    print("="*60)
    total = sum(results.values())
    for name, count in sorted(results.items(), key=lambda x: -x[1]):
        pct = count / total * 100
        bar = "█" * int(pct / 2)
        print(f"  {name}: {count} votes ({pct:.1f}%) {bar}")

    winner = max(results, key=results.get)
    print(f"\n  🏆 WINNER: {winner}")

    return results


print("✅ Phase 3 functions defined with Resume capability.")
print(f"   • Generated {len(VALUE_SETS)} unique value sets")
print("   • create_voter()")
print("   • generate_voters()")
print("   • voter_vote()")
print("   • run_election_with_resume() [NEW - with checkpoints]")
print("   • save_checkpoint() / load_checkpoint()")

✅ Phase 3 functions defined with Resume capability.
   • Generated 200 unique value sets
   • create_voter()
   • generate_voters()
   • voter_vote()
   • run_election_with_resume() [NEW - with checkpoints]
   • save_checkpoint() / load_checkpoint()


In [17]:
#@title Cell 7: Phase 4 - Analysis & Visualization

def plot_vote_results(results, candidates, scenario_name=""):
    """Bar chart showing vote counts per candidate."""
    names = [c["name"] for c in candidates]
    colors = ["#E63946", "#457B9D"]  # Red for Republican, Blue for Democrat
    votes = [results[n] for n in names]
    total = sum(votes)

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(names, votes, color=colors[:len(names)], edgecolor="white", width=0.5)

    # Add vote count + percentage labels on bars
    for bar, v in zip(bars, votes):
        pct = v / total * 100
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{v} votes\n({pct:.1f}%)", ha="center", va="bottom", fontsize=12, fontweight='bold')

    title = f"Election Results - {scenario_name}" if scenario_name else "Election Results"
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_ylabel("Number of Votes", fontsize=12)
    ax.set_ylim(0, max(votes) + max(votes)*0.2)
    ax.spines[["top", "right"]].set_visible(False)

    # Add winner annotation
    winner = max(results, key=results.get)
    ax.annotate(f"🏆 Winner: {winner}", xy=(0.5, 0.95), xycoords='axes fraction',
                ha='center', fontsize=12, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='gold', alpha=0.5))

    plt.tight_layout()
    filename = f"vote_results_{scenario_name.replace(' ', '_')}.png" if scenario_name else "vote_results.png"
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"📊 Chart saved as {filename}")


def plot_voter_traits(voters, scenario_name=""):
    """Box plot comparing trait distributions by vote choice."""
    records = [{**v["traits"], "vote": v["vote"]} for v in voters if v["vote"]]
    df = pd.DataFrame(records)

    if df.empty:
        print("⚠️ No votes recorded yet.")
        return

    traits = ["wisdom", "fear", "anger", "openness", "trust"]
    fig, axes = plt.subplots(1, len(traits), figsize=(16, 5), sharey=True)

    unique_votes = df["vote"].unique()
    colors = {"Donald Trump": "#E63946", "Kamala Harris": "#457B9D"}

    for ax, trait in zip(axes, traits):
        data = [df[df["vote"] == name][trait].values for name in unique_votes]
        bp = ax.boxplot(data, tick_labels=unique_votes, patch_artist=True)

        for patch, name in zip(bp['boxes'], unique_votes):
            patch.set_facecolor(colors.get(name, "#A8DADC"))
            patch.set_alpha(0.7)

        ax.set_title(trait.capitalize(), fontsize=11, fontweight='bold')
        ax.tick_params(axis="x", rotation=15)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_ylim(0, 11)

    title = f"Voter Traits by Choice - {scenario_name}" if scenario_name else "Voter Traits by Choice"
    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()

    filename = f"voter_traits_{scenario_name.replace(' ', '_')}.png" if scenario_name else "voter_traits.png"
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"📊 Trait chart saved as {filename}")


def plot_value_distribution(voters, scenario_name=""):
    """Analyze and plot value distribution by vote choice."""
    records = []
    for v in voters:
        if v["vote"]:
            # Classify values as conservative/progressive
            china_score = 1 if "aggressive" in v["values"]["china"].lower() or "military" in v["values"]["china"].lower() else 0
            health_score = 1 if "right" in v["values"]["healthcare"].lower() or "universal" in v["values"]["healthcare"].lower() else 0
            gun_score = 1 if "strict" in v["values"]["guns"].lower() or "ban" in v["values"]["guns"].lower() else 0

            records.append({
                "vote": v["vote"],
                "china_hawkish": china_score,
                "healthcare_progressive": health_score,
                "gun_control": gun_score
            })

    df = pd.DataFrame(records)

    if df.empty:
        print("⚠️ No votes recorded yet.")
        return

    # Group by vote and calculate means
    grouped = df.groupby("vote").mean()

    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(grouped.columns))
    width = 0.35

    colors = {"Donald Trump": "#E63946", "Kamala Harris": "#457B9D"}

    for i, (name, row) in enumerate(grouped.iterrows()):
        offset = width * (i - 0.5)
        bars = ax.bar(x + offset, row.values, width, label=name, color=colors.get(name, "#888"))

    ax.set_ylabel('Proportion of Voters')
    ax.set_title(f'Value Positions by Vote Choice - {scenario_name}' if scenario_name else 'Value Positions by Vote Choice')
    ax.set_xticks(x)
    ax.set_xticklabels(['China Hawkish', 'Healthcare Progressive', 'Gun Control Support'])
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    filename = f"value_dist_{scenario_name.replace(' ', '_')}.png" if scenario_name else "value_distribution.png"
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"📊 Value distribution chart saved as {filename}")


def analyze_results(results, voters, candidates, scenario_name=""):
    """Print a full textual analysis of the election outcome."""
    total = sum(results.values())

    print("\n" + "="*60)
    print(f" 📊 ELECTION RESULTS ANALYSIS {f'- {scenario_name}' if scenario_name else ''}")
    print("="*60)

    for name, count in sorted(results.items(), key=lambda x: -x[1]):
        bar = "█" * int(count / total * 30)
        print(f"  {name:20s}: {count:3d} votes ({count/total*100:5.1f}%) {bar}")

    winner = max(results, key=results.get)
    margin = results[winner] - min(results.values())
    print(f"\n  🏆 WINNER: {winner}")
    print(f"  📈 Margin: {margin} votes ({margin/total*100:.1f}%)")

    # Trait averages per candidate
    print("\n" + "-"*60)
    print(" 📋 Average Traits of Voters by Choice")
    print("-"*60)

    records = [{**v["traits"], "vote": v["vote"]} for v in voters if v["vote"]]
    df = pd.DataFrame(records)

    if not df.empty:
        summary = df.groupby("vote").agg(['mean', 'std']).round(2)

        for candidate in df["vote"].unique():
            print(f"\n  {candidate}:")
            cand_data = df[df["vote"] == candidate]
            for trait in ["wisdom", "fear", "anger", "openness", "trust"]:
                mean_val = cand_data[trait].mean()
                std_val = cand_data[trait].std()
                print(f"    {trait:10s}: {mean_val:.2f} ± {std_val:.2f}")

    # Sample explanations
    print("\n" + "-"*60)
    print(" 💬 Sample Voter Explanations")
    print("-"*60)

    for candidate in results.keys():
        cand_voters = [v for v in voters if v["vote"] == candidate and v.get("explanation")]
        if cand_voters:
            sample = random.choice(cand_voters)
            print(f"\n  [{candidate} voter #{sample['id']}]:")
            print(f"    Traits: W:{sample['traits']['wisdom']} F:{sample['traits']['fear']} A:{sample['traits']['anger']}")
            print(f"    Reason: {sample['explanation'][:200]}...")

    print("\n" + "="*60)


def save_results(results, voters, scenario_name):
    """Save results to JSON for later analysis."""
    data = {
        "scenario": scenario_name,
        "results": results,
        "voters": [
            {
                "id": v["id"],
                "traits": v["traits"],
                "values": v["values"],
                "vote": v["vote"],
                "explanation": v.get("explanation", "")
            }
            for v in voters
        ]
    }

    filename = f"results_{scenario_name.replace(' ', '_')}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"💾 Results saved to {filename}")


print("✅ Phase 4 functions defined.")
print("   • plot_vote_results()")
print("   • plot_voter_traits()")
print("   • plot_value_distribution()")
print("   • analyze_results()")
print("   • save_results()")

✅ Phase 4 functions defined.
   • plot_vote_results()
   • plot_voter_traits()
   • plot_value_distribution()
   • analyze_results()
   • save_results()


In [18]:
#@title Cell 8: Define Scenarios (1-5)

# ══════════════════════════════════════════════════════════════
# SCENARIO DEFINITIONS (as per project requirements)
# ══════════════════════════════════════════════════════════════

SCENARIOS = {
    "Scenario 1": {
        "name": "Low Wisdom, Mixed Fear",
        "description": "Majority low-wisdom voters with mixed fear levels",
        "n_voters": 200,
        "wisdom_dist": {"low": 0.60, "medium": 0.30, "high": 0.10},
        "fear_dist": {"low": 0.30, "medium": 0.40, "high": 0.30}
    },
    "Scenario 2": {
        "name": "High Wisdom, Low Fear",
        "description": "Educated, calm electorate",
        "n_voters": 200,
        "wisdom_dist": {"low": 0.10, "medium": 0.30, "high": 0.60},
        "fear_dist": {"low": 0.60, "medium": 0.30, "high": 0.10}
    },
    "Scenario 3": {
        "name": "High Fear, Mixed Wisdom",
        "description": "Anxious electorate with varied education",
        "n_voters": 200,
        "wisdom_dist": {"low": 0.33, "medium": 0.34, "high": 0.33},
        "fear_dist": {"low": 0.10, "medium": 0.30, "high": 0.60}
    },
    "Scenario 4": {
        "name": "Balanced Population",
        "description": "Evenly distributed traits",
        "n_voters": 200,
        "wisdom_dist": {"low": 0.33, "medium": 0.34, "high": 0.33},
        "fear_dist": {"low": 0.33, "medium": 0.34, "high": 0.33}
    },
    "Scenario 5": {
        "name": "Polarized Population",
        "description": "Extreme traits, few moderates",
        "n_voters": 200,
        "wisdom_dist": {"low": 0.45, "medium": 0.10, "high": 0.45},
        "fear_dist": {"low": 0.45, "medium": 0.10, "high": 0.45}
    }
}

# For testing with lower rate limits, use reduced voter counts
SCENARIOS_LITE = {
    key: {**val, "n_voters": 20}
    for key, val in SCENARIOS.items()
}

print("✅ Scenarios defined:")
print("-" * 60)
for key, scenario in SCENARIOS.items():
    print(f"\n📋 {key}: {scenario['name']}")
    print(f"   {scenario['description']}")
    print(f"   Voters: {scenario['n_voters']}")
    print(f"   Wisdom: L={scenario['wisdom_dist']['low']:.0%}, M={scenario['wisdom_dist']['medium']:.0%}, H={scenario['wisdom_dist']['high']:.0%}")
    print(f"   Fear:   L={scenario['fear_dist']['low']:.0%}, M={scenario['fear_dist']['medium']:.0%}, H={scenario['fear_dist']['high']:.0%}")

print("\n" + "-" * 60)
print("💡 Use SCENARIOS for full run (200 voters)")
print("💡 Use SCENARIOS_LITE for testing (20 voters)")

✅ Scenarios defined:
------------------------------------------------------------

📋 Scenario 1: Low Wisdom, Mixed Fear
   Majority low-wisdom voters with mixed fear levels
   Voters: 200
   Wisdom: L=60%, M=30%, H=10%
   Fear:   L=30%, M=40%, H=30%

📋 Scenario 2: High Wisdom, Low Fear
   Educated, calm electorate
   Voters: 200
   Wisdom: L=10%, M=30%, H=60%
   Fear:   L=60%, M=30%, H=10%

📋 Scenario 3: High Fear, Mixed Wisdom
   Anxious electorate with varied education
   Voters: 200
   Wisdom: L=33%, M=34%, H=33%
   Fear:   L=10%, M=30%, H=60%

📋 Scenario 4: Balanced Population
   Evenly distributed traits
   Voters: 200
   Wisdom: L=33%, M=34%, H=33%
   Fear:   L=33%, M=34%, H=33%

📋 Scenario 5: Polarized Population
   Extreme traits, few moderates
   Voters: 200
   Wisdom: L=45%, M=10%, H=45%
   Fear:   L=45%, M=10%, H=45%

------------------------------------------------------------
💡 Use SCENARIOS for full run (200 voters)
💡 Use SCENARIOS_LITE for testing (20 voters)


In [ ]:
#@title Cell 9: FAST Runner - 200 Voters

import concurrent.futures
import threading

# ══════════════════════════════════════════════════════════════
# CONFIGURATION - OPTIMIZED FOR SPEED
# ══════════════════════════════════════════════════════════════

N_VOTERS = 200                # تعداد رأی‌دهندگان
SELECTED_SCENARIO = "Scenario 1"
QUESTIONS_PER_TOPIC = 5
USE_DEBATE_CACHE = True

# ═══ Speed Settings ═══
BATCH_SIZE = 10               # افزایش از 3 به 10
BATCH_DELAY = 45              # کاهش از 90 به 45 ثانیه
CHECKPOINT_EVERY = 10         # ذخیره هر 10 رأی‌دهنده
DELAY_BETWEEN_VOTERS = 0.5    # کاهش از 2 به 0.5 ثانیه

# زمان تخمینی: ~30-40 دقیقه برای 200 رأی‌دهنده

# ══════════════════════════════════════════════════════════════
# FAST VOTER FUNCTION
# ══════════════════════════════════════════════════════════════

def voter_vote_fast(voter, transcript_summary, candidates):
    """
    نسخه سریع‌تر با prompt کوتاه‌تر
    """
    cand_names = [c["name"] for c in candidates]

    # Prompt کوتاه‌تر = توکن کمتر = سرعت بیشتر
    traits = voter["traits"]
    prompt = (
        f"You're a voter. Wisdom:{traits['wisdom']}/10, Fear:{traits['fear']}/10, "
        f"Anger:{traits['anger']}/10.\n"
        f"Values: {voter['values']['guns'][:50]}...\n"
        f"Debate summary: {transcript_summary[:1500]}\n\n"
        f"Vote for Donald Trump or Kamala Harris?\n"
        f"Reply: [1-2 sentence reason] VOTE: [name]"
    )

    raw = call_model(SMALL_MODEL, prompt, max_tokens=100)  # کاهش از 200 به 100

    vote = None
    for name in cand_names:
        if name.lower() in raw.lower():
            vote = name
            break

    if vote is None:
        vote = random.choice(cand_names)

    explanation = raw.split("VOTE:")[0].strip()[:150] if "VOTE:" in raw else raw[:100]

    return vote, explanation


def run_election_fast(voters, transcript, candidates,
                      batch_size=10, delay_sec=45, checkpoint_every=10):
    """
    نسخه سریع با تنظیمات بهینه
    """

    # Check checkpoint
    checkpoint = load_checkpoint()

    if checkpoint and len(checkpoint["voters"]) == len(voters):
        start_idx = checkpoint["last_voter_idx"] + 1
        results = checkpoint["results"]

        for i, v_data in enumerate(checkpoint["voters"]):
            if v_data.get("vote"):
                voters[i]["vote"] = v_data["vote"]
                voters[i]["explanation"] = v_data.get("explanation")

        completed = sum(1 for v in voters if v.get("vote"))
        print(f"📂 Resuming: {completed}/{len(voters)} already done")
    else:
        start_idx = 0
        results = {c["name"]: 0 for c in candidates}
        if checkpoint:
            clear_checkpoint()

    total = len(voters)
    remaining = total - start_idx
    total_batches = (remaining - 1) // batch_size + 1
    est_time = (total_batches * delay_sec + remaining * DELAY_BETWEEN_VOTERS) / 60

    print(f"\n🚀 FAST Election Mode")
    print(f"   Remaining: {remaining}/{total} voters")
    print(f"   Batch: {batch_size} voters, {delay_sec}s delay")
    print(f"   Estimated: ~{est_time:.0f} minutes\n")

    for i in range(start_idx, total):
        voter = voters[i]

        if voter.get("vote"):
            continue

        # Batch header
        batch_num = (i - start_idx) // batch_size + 1
        if (i - start_idx) % batch_size == 0:
            print(f"\n📦 Batch {batch_num}/{total_batches}")
            print("-" * 50)

        try:
            choice, explanation = voter_vote_fast(voter, transcript, candidates)
            voter["vote"] = choice
            voter["explanation"] = explanation
            results[choice] += 1

            # Compact output
            w, f = voter['traits']['wisdom'], voter['traits']['fear']
            print(f"  V{voter['id']:>3} [W:{w} F:{f}] → {choice.split()[1]}")  # فقط نام خانوادگی

            time.sleep(DELAY_BETWEEN_VOTERS)

        except Exception as e:
            print(f"\n  ⚠️ Error V{voter['id']}: {str(e)[:80]}")
            save_checkpoint(voters, results, i - 1)
            print(f"  ⏳ Waiting 60s...")
            time.sleep(60)

            try:
                choice, explanation = voter_vote_fast(voter, transcript, candidates)
                voter["vote"] = choice
                voter["explanation"] = explanation
                results[choice] += 1
                print(f"  ✅ Retry OK: V{voter['id']} → {choice.split()[1]}")
            except:
                print(f"  ❌ Skip V{voter['id']}")
                continue

        # Checkpoint
        if (i + 1) % checkpoint_every == 0:
            save_checkpoint(voters, results, i)

            # Quick progress
            total_votes = sum(results.values())
            pcts = {n: f"{c/total_votes*100:.0f}%" for n, c in results.items()}
            print(f"  💾 [{total_votes}/{total}] {pcts}")

        # Batch delay
        if (i - start_idx + 1) % batch_size == 0 and i < total - 1:
            print(f"\n  ⏳ {delay_sec}s pause...")
            time.sleep(delay_sec)

    clear_checkpoint()

    # Final
    print("\n" + "="*60)
    print(" ✅ ELECTION COMPLETE")
    print("="*60)

    total_votes = sum(results.values())
    for name, count in sorted(results.items(), key=lambda x: -x[1]):
        pct = count / total_votes * 100
        bar = "█" * int(pct / 2)
        print(f"  {name}: {count} ({pct:.1f}%) {bar}")

    print(f"\n  🏆 WINNER: {max(results, key=results.get)}")

    return results


# ══════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ══════════════════════════════════════════════════════════════

scenario = SCENARIOS[SELECTED_SCENARIO]

print("=" * 70)
print(" 🗳️ FAST ELECTION SIMULATION - 200 VOTERS")
print("=" * 70)
print(f"\n📋 {SELECTED_SCENARIO}: {scenario['name']}")
print(f"   Voters: {N_VOTERS}")

# Phase 2: Debate
print("\n" + "="*70)
print(" 🎤 PHASE 2: Debate")
print("="*70)

transcript, final_context = get_or_run_debate(
    CANDIDATES, DEBATE_TOPICS, QUESTIONS_PER_TOPIC, use_cache=USE_DEBATE_CACHE
)
print(f"✅ Transcript: {len(transcript):,} chars")

# Phase 3: Election
print("\n" + "="*70)
print(f" 🗳️ PHASE 3: Election ({N_VOTERS} voters)")
print("="*70)

# Generate/Load voters
checkpoint = load_checkpoint()
if checkpoint and len(checkpoint["voters"]) == N_VOTERS:
    voters = []
    for v_data in checkpoint["voters"]:
        v = create_voter(v_data["id"], v_data["traits"], v_data["values"])
        v["vote"] = v_data.get("vote")
        v["explanation"] = v_data.get("explanation")
        voters.append(v)
    print(f"📂 Loaded {N_VOTERS} voters from checkpoint")
else:
    voters = generate_voters(
        N_VOTERS,
        wisdom_dist=scenario['wisdom_dist'],
        fear_dist=scenario['fear_dist']
    )
    print(f"✅ Generated {N_VOTERS} voters")
    if checkpoint:
        clear_checkpoint()

# Quick stats
df = pd.DataFrame([v["traits"] for v in voters])
print(f"\n📊 Traits: Wisdom={df['wisdom'].mean():.1f}±{df['wisdom'].std():.1f}, "
      f"Fear={df['fear'].mean():.1f}±{df['fear'].std():.1f}")

# Run
results = run_election_fast(
    voters, transcript, CANDIDATES,
    batch_size=BATCH_SIZE,
    delay_sec=BATCH_DELAY,
    checkpoint_every=CHECKPOINT_EVERY
)

# Phase 4: Analysis
print("\n" + "="*70)
print(" 📊 PHASE 4: Analysis")
print("="*70)

analyze_results(results, voters, CANDIDATES, scenario['name'])
plot_vote_results(results, CANDIDATES, scenario['name'])
plot_voter_traits(voters, scenario['name'])

# Save
save_all_data(CANDIDATES, transcript, final_context, voters, results, SELECTED_SCENARIO)

print("\n" + "="*70)
print(" ✅ COMPLETE")
print("="*70)

 🗳️ FAST ELECTION SIMULATION - 200 VOTERS

📋 Scenario 1: Low Wisdom, Mixed Fear
   Voters: 200

 🎤 PHASE 2: Debate
📂 Loading debate from cache...
   Loaded transcript: 51127 chars
✅ Transcript: 51,127 chars

 🗳️ PHASE 3: Election (200 voters)
✅ Generated 200 voters
  🗑️ Checkpoint cleared

📊 Traits: Wisdom=3.5±2.4, Fear=5.1±2.9

🚀 FAST Election Mode
   Remaining: 200/200 voters
   Batch: 10 voters, 45s delay
   Estimated: ~17 minutes


📦 Batch 1/20
--------------------------------------------------
  ⚠️ 503 (Server Busy) - Attempt 1/8
  ⏳ Waiting 16s (base: 20s)...
  V  1 [W:1 F:3] → Harris
  V  2 [W:6 F:7] → Trump
  V  3 [W:10 F:5] → Trump
  V  4 [W:2 F:4] → Trump
  V  5 [W:6 F:5] → Trump
  V  6 [W:3 F:6] → Trump
  V  7 [W:8 F:7] → Trump
  V  8 [W:2 F:3] → Trump
  V  9 [W:3 F:9] → Harris
  V 10 [W:3 F:10] → Trump
  💾 Checkpoint saved at voter 10
  💾 [10/200] {'Donald Trump': '80%', 'Kamala Harris': '20%'}

  ⏳ 45s pause...

📦 Batch 2/20
------------------------------------------------

In [ ]:
#@title Cell 10: Run All Scenarios (Full Comparison)

# ══════════════════════════════════════════════════════════════
# RUN ALL 5 SCENARIOS AND COMPARE
# ══════════════════════════════════════════════════════════════

# Use LITE for testing
USE_LITE = True
scenarios = SCENARIOS_LITE if USE_LITE else SCENARIOS

# Store all results
all_results = {}

print("=" * 70)
print(" 🗳️ MULTI-SCENARIO ELECTION SIMULATION")
print("=" * 70)
print(f"\n📋 Running {len(scenarios)} scenarios...")
print(f"   Mode: {'LITE (20 voters each)' if USE_LITE else 'FULL (200 voters each)'}")

# First, ensure we have the debate
print("\n" + "="*70)
print(" 🎤 Loading/Running Debate (shared across all scenarios)")
print("="*70)

transcript, _ = get_or_run_debate(CANDIDATES, DEBATE_TOPICS, 5, use_cache=True)

# Run each scenario
for scenario_key, scenario in scenarios.items():
    print("\n" + "="*70)
    print(f" 🗳️ {scenario_key}: {scenario['name']}")
    print("="*70)

    # Generate voters for this scenario
    voters = generate_voters(
        scenario['n_voters'],
        wisdom_dist=scenario['wisdom_dist'],
        fear_dist=scenario['fear_dist']
    )

    # Run election
    results = run_election_batched(
        voters, transcript, CANDIDATES,
        batch_size=BATCH_SIZE, delay_sec=BATCH_DELAY
    )

    # Store results
    all_results[scenario_key] = {
        "scenario": scenario,
        "results": results,
        "voters": voters
    }

    # Quick summary
    winner = max(results, key=results.get)
    total = sum(results.values())
    print(f"\n🏆 Winner: {winner} ({results[winner]/total*100:.1f}%)")

    # Save individual results
    save_results(results, voters, scenario_key)

# ══════════════════════════════════════════════════════════════
# COMPARATIVE ANALYSIS
# ══════════════════════════════════════════════════════════════

print("\n" + "="*70)
print(" 📊 COMPARATIVE ANALYSIS - ALL SCENARIOS")
print("="*70)

# Create comparison table
comparison_data = []
for scenario_key, data in all_results.items():
    results = data["results"]
    total = sum(results.values())

    comparison_data.append({
        "Scenario": scenario_key,
        "Name": data["scenario"]["name"],
        "Trump Votes": results.get("Donald Trump", 0),
        "Trump %": results.get("Donald Trump", 0) / total * 100,
        "Harris Votes": results.get("Kamala Harris", 0),
        "Harris %": results.get("Kamala Harris", 0) / total * 100,
        "Winner": max(results, key=results.get)
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n")
print(comparison_df.to_string(index=False))

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df["Trump %"], width,
               label='Donald Trump', color='#E63946')
bars2 = ax.bar(x + width/2, comparison_df["Harris %"], width,
               label='Kamala Harris', color='#457B9D')

ax.set_ylabel('Vote Percentage (%)')
ax.set_title('Election Results Across All Scenarios', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"{row['Scenario']}\n{row['Name']}" for _, row in comparison_df.iterrows()],
                   fontsize=9)
ax.legend()
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% line')
ax.set_ylim(0, 100)
ax.spines[["top", "right"]].set_visible(False)

# Add percentage labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig("all_scenarios_comparison.png", dpi=150)
plt.show()

print("\n📊 Comparison chart saved as all_scenarios_comparison.png")

# ══════════════════════════════════════════════════════════════
# KEY INSIGHTS
# ══════════════════════════════════════════════════════════════

print("\n" + "="*70)
print(" 💡 KEY INSIGHTS")
print("="*70)

# Find patterns
trump_wins = comparison_df[comparison_df["Winner"] == "Donald Trump"]
harris_wins = comparison_df[comparison_df["Winner"] == "Kamala Harris"]

print(f"\n📈 Trump won {len(trump_wins)}/{len(comparison_df)} scenarios")
print(f"📈 Harris won {len(harris_wins)}/{len(comparison_df)} scenarios")

# Best scenario for each candidate
best_trump = comparison_df.loc[comparison_df["Trump %"].idxmax()]
best_harris = comparison_df.loc[comparison_df["Harris %"].idxmax()]

print(f"\n🔴 Best scenario for Trump: {best_trump['Scenario']} ({best_trump['Name']}) - {best_trump['Trump %']:.1f}%")
print(f"🔵 Best scenario for Harris: {best_harris['Scenario']} ({best_harris['Name']}) - {best_harris['Harris %']:.1f}%")

print("\n" + "="*70)
print(" ✅ ALL SCENARIOS COMPLETE")
print("="*70)

In [ ]:
#@title Cell 11: Save All Data to Files

import json
import pickle
import os
from datetime import datetime

# ══════════════════════════════════════════════════════════════
# SAVE ALL SIMULATION DATA
# ══════════════════════════════════════════════════════════════

def save_all_data(candidates, transcript, context, voters, results,
                  scenario_name="default", output_dir="simulation_data"):
    """
    Save all simulation data to files for later use without API calls.

    Creates:
    - candidates.json: Candidate information and personas
    - debate_transcript.json: Full debate transcript and context
    - voters.json: All voter data with traits, values, votes, explanations
    - results.json: Election results summary
    - full_simulation.pkl: Complete pickle backup of all data
    - metadata.json: Simulation metadata and timestamps
    """

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    print("="*60)
    print(" 💾 SAVING ALL SIMULATION DATA")
    print("="*60)
    print(f"📁 Output directory: {output_dir}/")
    print(f"📅 Timestamp: {timestamp}")
    print()

    # ── 1. Save Candidates ────────────────────────────────────
    candidates_data = []
    for c in candidates:
        candidates_data.append({
            "name": c["name"],
            "party": c["party"],
            "traits": c["traits"],
            "persona": c["persona"]
        })

    candidates_file = os.path.join(output_dir, "candidates.json")
    with open(candidates_file, 'w', encoding='utf-8') as f:
        json.dump(candidates_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved: candidates.json ({len(candidates)} candidates)")

    # ── 2. Save Debate Transcript ─────────────────────────────
    debate_data = {
        "transcript": transcript,
        "context": context,
        "topics": DEBATE_TOPICS,
        "questions_per_topic": QUESTIONS_PER_TOPIC if 'QUESTIONS_PER_TOPIC' in dir() else 5
    }

    debate_file = os.path.join(output_dir, "debate_transcript.json")
    with open(debate_file, 'w', encoding='utf-8') as f:
        json.dump(debate_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved: debate_transcript.json ({len(transcript):,} chars)")

    # ── 3. Save Voters ────────────────────────────────────────
    voters_data = []
    for v in voters:
        voters_data.append({
            "id": v["id"],
            "traits": v["traits"],
            "values": v["values"],
            "vote": v.get("vote"),
            "explanation": v.get("explanation", "")
        })

    voters_file = os.path.join(output_dir, "voters.json")
    with open(voters_file, 'w', encoding='utf-8') as f:
        json.dump(voters_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved: voters.json ({len(voters)} voters)")

    # ── 4. Save Results ───────────────────────────────────────
    total_votes = sum(results.values())
    results_data = {
        "scenario": scenario_name,
        "results": results,
        "total_votes": total_votes,
        "percentages": {k: round(v/total_votes*100, 2) for k, v in results.items()},
        "winner": max(results, key=results.get),
        "margin": max(results.values()) - min(results.values())
    }

    results_file = os.path.join(output_dir, "results.json")
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(results_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved: results.json (Winner: {results_data['winner']})")

    # ── 5. Save Metadata ──────────────────────────────────────
    metadata = {
        "timestamp": timestamp,
        "scenario": scenario_name,
        "n_candidates": len(candidates),
        "n_voters": len(voters),
        "n_topics": len(DEBATE_TOPICS),
        "topics": DEBATE_TOPICS,
        "transcript_length": len(transcript),
        "models_used": {
            "large": LARGE_MODEL if 'LARGE_MODEL' in dir() else "unknown",
            "small": SMALL_MODEL if 'SMALL_MODEL' in dir() else "unknown"
        },
        "files_saved": [
            "candidates.json",
            "debate_transcript.json",
            "voters.json",
            "results.json",
            "metadata.json",
            "full_simulation.pkl"
        ]
    }

    metadata_file = os.path.join(output_dir, "metadata.json")
    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved: metadata.json")

    # ── 6. Save Complete Pickle Backup ────────────────────────
    full_data = {
        "candidates": candidates,
        "transcript": transcript,
        "context": context,
        "voters": voters,
        "results": results,
        "metadata": metadata
    }

    pickle_file = os.path.join(output_dir, "full_simulation.pkl")
    with open(pickle_file, 'wb') as f:
        pickle.dump(full_data, f)
    print(f"✅ Saved: full_simulation.pkl (complete backup)")

    # ── Summary ───────────────────────────────────────────────
    print()
    print("="*60)
    print(" ✅ ALL DATA SAVED SUCCESSFULLY")
    print("="*60)
    print(f"📁 Location: {output_dir}/")
    print(f"📊 Files created: {len(metadata['files_saved'])}")

    # Calculate total size
    total_size = 0
    for filename in os.listdir(output_dir):
        filepath = os.path.join(output_dir, filename)
        total_size += os.path.getsize(filepath)
    print(f"💾 Total size: {total_size/1024:.1f} KB")

    return output_dir


def save_multi_scenario_data(all_results, transcript, context, candidates,
                             output_dir="multi_scenario_data"):
    """
    Save data from multiple scenarios for comparison analysis.
    """
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    print("="*60)
    print(" 💾 SAVING MULTI-SCENARIO DATA")
    print("="*60)

    # Save shared debate
    debate_data = {
        "transcript": transcript,
        "context": context,
        "topics": DEBATE_TOPICS
    }
    with open(os.path.join(output_dir, "debate_transcript.json"), 'w', encoding='utf-8') as f:
        json.dump(debate_data, f, ensure_ascii=False, indent=2)
    print("✅ Saved: debate_transcript.json")

    # Save candidates
    candidates_data = [{"name": c["name"], "party": c["party"], "traits": c["traits"], "persona": c["persona"]} for c in candidates]
    with open(os.path.join(output_dir, "candidates.json"), 'w', encoding='utf-8') as f:
        json.dump(candidates_data, f, ensure_ascii=False, indent=2)
    print("✅ Saved: candidates.json")

    # Save each scenario
    scenarios_summary = []
    for scenario_key, data in all_results.items():
        scenario_dir = os.path.join(output_dir, scenario_key.replace(" ", "_"))
        os.makedirs(scenario_dir, exist_ok=True)

        # Save voters
        voters_data = [{
            "id": v["id"],
            "traits": v["traits"],
            "values": v["values"],
            "vote": v.get("vote"),
            "explanation": v.get("explanation", "")
        } for v in data["voters"]]

        with open(os.path.join(scenario_dir, "voters.json"), 'w', encoding='utf-8') as f:
            json.dump(voters_data, f, ensure_ascii=False, indent=2)

        # Save results
        results = data["results"]
        total = sum(results.values())
        results_data = {
            "scenario": scenario_key,
            "config": data["scenario"],
            "results": results,
            "percentages": {k: round(v/total*100, 2) for k, v in results.items()},
            "winner": max(results, key=results.get)
        }

        with open(os.path.join(scenario_dir, "results.json"), 'w', encoding='utf-8') as f:
            json.dump(results_data, f, ensure_ascii=False, indent=2)

        scenarios_summary.append(results_data)
        print(f"✅ Saved: {scenario_key}/ (voters + results)")

    # Save summary
    with open(os.path.join(output_dir, "all_scenarios_summary.json"), 'w', encoding='utf-8') as f:
        json.dump(scenarios_summary, f, ensure_ascii=False, indent=2)
    print("✅ Saved: all_scenarios_summary.json")

    # Save complete pickle
    with open(os.path.join(output_dir, "full_multi_scenario.pkl"), 'wb') as f:
        pickle.dump({"all_results": all_results, "transcript": transcript, "candidates": candidates}, f)
    print("✅ Saved: full_multi_scenario.pkl")

    print(f"\n📁 All data saved to: {output_dir}/")
    return output_dir


# ══════════════════════════════════════════════════════════════
# EXECUTE SAVE (if data exists in memory)
# ══════════════════════════════════════════════════════════════

# Check if we have data to save
if 'voters' in dir() and 'results' in dir() and 'transcript' in dir():
    print("📊 Found simulation data in memory. Saving...")
    scenario_name = SELECTED_SCENARIO if 'SELECTED_SCENARIO' in dir() else "default"
    save_all_data(CANDIDATES, transcript, final_context, voters, results, scenario_name)
else:
    print("⚠️ No simulation data found in memory.")
    print("   Run the simulation first (Cell 9), then run this cell to save.")

print("\n✅ Save functions defined:")
print("   • save_all_data() - Save single scenario")
print("   • save_multi_scenario_data() - Save all scenarios")

In [ ]:
#@title Cell 12: Load All Data from Files

import json
import pickle
import os

# ══════════════════════════════════════════════════════════════
# LOAD ALL SIMULATION DATA
# ══════════════════════════════════════════════════════════════

def load_all_data(input_dir="simulation_data"):
    """
    Load all simulation data from files.
    Returns dict with: candidates, transcript, context, voters, results, metadata
    """

    print("="*60)
    print(" 📂 LOADING SIMULATION DATA")
    print("="*60)
    print(f"📁 Input directory: {input_dir}/")
    print()

    if not os.path.exists(input_dir):
        print(f"❌ Directory not found: {input_dir}")
        return None

    data = {}

    # ── 1. Load from Pickle (fastest, complete) ───────────────
    pickle_file = os.path.join(input_dir, "full_simulation.pkl")
    if os.path.exists(pickle_file):
        print("🔄 Loading from pickle backup...")
        with open(pickle_file, 'rb') as f:
            data = pickle.load(f)
        print(f"✅ Loaded: full_simulation.pkl")
        print(f"   • Candidates: {len(data['candidates'])}")
        print(f"   • Voters: {len(data['voters'])}")
        print(f"   • Transcript: {len(data['transcript']):,} chars")
        print()
        print("="*60)
        print(" ✅ DATA LOADED SUCCESSFULLY (from pickle)")
        print("="*60)
        return data

    # ── 2. Load from JSON files (fallback) ────────────────────
    print("🔄 Loading from JSON files...")

    # Load candidates
    candidates_file = os.path.join(input_dir, "candidates.json")
    if os.path.exists(candidates_file):
        with open(candidates_file, 'r', encoding='utf-8') as f:
            data['candidates'] = json.load(f)
        print(f"✅ Loaded: candidates.json ({len(data['candidates'])} candidates)")

    # Load debate
    debate_file = os.path.join(input_dir, "debate_transcript.json")
    if os.path.exists(debate_file):
        with open(debate_file, 'r', encoding='utf-8') as f:
            debate_data = json.load(f)
            data['transcript'] = debate_data['transcript']
            data['context'] = debate_data['context']
        print(f"✅ Loaded: debate_transcript.json ({len(data['transcript']):,} chars)")

    # Load voters
    voters_file = os.path.join(input_dir, "voters.json")
    if os.path.exists(voters_file):
        with open(voters_file, 'r', encoding='utf-8') as f:
            data['voters'] = json.load(f)
        print(f"✅ Loaded: voters.json ({len(data['voters'])} voters)")

    # Load results
    results_file = os.path.join(input_dir, "results.json")
    if os.path.exists(results_file):
        with open(results_file, 'r', encoding='utf-8') as f:
            results_data = json.load(f)
            data['results'] = results_data['results']
        print(f"✅ Loaded: results.json")

    # Load metadata
    metadata_file = os.path.join(input_dir, "metadata.json")
    if os.path.exists(metadata_file):
        with open(metadata_file, 'r', encoding='utf-8') as f:
            data['metadata'] = json.load(f)
        print(f"✅ Loaded: metadata.json")

    print()
    print("="*60)
    print(" ✅ DATA LOADED SUCCESSFULLY (from JSON)")
    print("="*60)

    return data


def load_multi_scenario_data(input_dir="multi_scenario_data"):
    """
    Load multi-scenario data from files.
    Returns dict with all scenarios and shared data.
    """

    print("="*60)
    print(" 📂 LOADING MULTI-SCENARIO DATA")
    print("="*60)

    if not os.path.exists(input_dir):
        print(f"❌ Directory not found: {input_dir}")
        return None

    # Try pickle first
    pickle_file = os.path.join(input_dir, "full_multi_scenario.pkl")
    if os.path.exists(pickle_file):
        print("🔄 Loading from pickle...")
        with open(pickle_file, 'rb') as f:
            data = pickle.load(f)
        print("✅ Loaded complete multi-scenario data")
        return data

    # Load from JSON
    data = {"all_results": {}}

    # Load shared debate
    debate_file = os.path.join(input_dir, "debate_transcript.json")
    if os.path.exists(debate_file):
        with open(debate_file, 'r', encoding='utf-8') as f:
            debate_data = json.load(f)
            data['transcript'] = debate_data['transcript']
        print("✅ Loaded: debate_transcript.json")

    # Load candidates
    candidates_file = os.path.join(input_dir, "candidates.json")
    if os.path.exists(candidates_file):
        with open(candidates_file, 'r', encoding='utf-8') as f:
            data['candidates'] = json.load(f)
        print("✅ Loaded: candidates.json")

    # Load summary
    summary_file = os.path.join(input_dir, "all_scenarios_summary.json")
    if os.path.exists(summary_file):
        with open(summary_file, 'r', encoding='utf-8') as f:
            data['summary'] = json.load(f)
        print("✅ Loaded: all_scenarios_summary.json")

    # Load each scenario
    for item in os.listdir(input_dir):
        item_path = os.path.join(input_dir, item)
        if os.path.isdir(item_path):
            scenario_key = item.replace("_", " ")

            voters_file = os.path.join(item_path, "voters.json")
            results_file = os.path.join(item_path, "results.json")

            if os.path.exists(voters_file) and os.path.exists(results_file):
                with open(voters_file, 'r', encoding='utf-8') as f:
                    voters = json.load(f)
                with open(results_file, 'r', encoding='utf-8') as f:
                    results_data = json.load(f)

                data['all_results'][scenario_key] = {
                    "voters": voters,
                    "results": results_data['results'],
                    "scenario": results_data.get('config', {})
                }
                print(f"✅ Loaded: {item}/")

    print()
    print("="*60)
    print(f" ✅ LOADED {len(data['all_results'])} SCENARIOS")
    print("="*60)

    return data


def quick_load(input_dir="simulation_data"):
    """
    Quick load and assign to global variables for immediate use.
    """
    data = load_all_data(input_dir)

    if data:
        print("\n📌 Assigning to global variables...")
        print("   • loaded_candidates")
        print("   • loaded_transcript")
        print("   • loaded_voters")
        print("   • loaded_results")
        return data
    return None


# ══════════════════════════════════════════════════════════════
# EXECUTE LOAD
# ══════════════════════════════════════════════════════════════

# Try to load existing data
if os.path.exists("simulation_data"):
    loaded_data = quick_load("simulation_data")
    if loaded_data:
        loaded_candidates = loaded_data.get('candidates', [])
        loaded_transcript = loaded_data.get('transcript', '')
        loaded_voters = loaded_data.get('voters', [])
        loaded_results = loaded_data.get('results', {})
        loaded_context = loaded_data.get('context', '')
else:
    print("⚠️ No saved data found.")
    print("   Run simulation (Cell 9) and save (Cell 11) first.")

print("\n✅ Load functions defined:")
print("   • load_all_data() - Load single scenario")
print("   • load_multi_scenario_data() - Load all scenarios")
print("   • quick_load() - Load and assign to variables

In [ ]:
#@title Cell 13: Comprehensive Report Charts (All in One Image)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

# ══════════════════════════════════════════════════════════════
# COMPREHENSIVE REPORT VISUALIZATION
# ══════════════════════════════════════════════════════════════

def create_comprehensive_report(voters, results, candidates, scenario_name="",
                                 save_path="comprehensive_report.png", dpi=200):
    """
    Create a comprehensive multi-panel report with all key visualizations.
    All charts in one image for easy inclusion in reports.
    """

    # Prepare data
    records = [{**v["traits"], "vote": v["vote"]} for v in voters if v.get("vote")]
    df = pd.DataFrame(records)

    if df.empty:
        print("❌ No vote data available")
        return

    # Define colors
    colors = {
        "Donald Trump": "#E63946",
        "Kamala Harris": "#457B9D"
    }

    # Create figure with GridSpec for flexible layout
    fig = plt.figure(figsize=(20, 16))
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.35, wspace=0.3)

    # Title
    title = f"Election Simulation Report"
    if scenario_name:
        title += f" - {scenario_name}"
    fig.suptitle(title, fontsize=20, fontweight='bold', y=0.98)

    # ══════════════════════════════════════════════════════════
    # Panel 1: Election Results (Bar Chart)
    # ══════════════════════════════════════════════════════════
    ax1 = fig.add_subplot(gs[0, 0:2])

    names = [c["name"] for c in candidates]
    votes = [results.get(n, 0) for n in names]
    total = sum(votes)

    bars = ax1.bar(names, votes, color=[colors.get(n, '#888') for n in names],
                   edgecolor='white', width=0.6)

    for bar, v in zip(bars, votes):
        pct = v / total * 100
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{v} votes\n({pct:.1f}%)', ha='center', va='bottom',
                fontsize=12, fontweight='bold')

    winner = max(results, key=results.get)
    ax1.set_title(f'🏆 Election Results (Winner: {winner})', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Number of Votes', fontsize=11)
    ax1.set_ylim(0, max(votes) * 1.25)
    ax1.spines[['top', 'right']].set_visible(False)

    # ══════════════════════════════════════════════════════════
    # Panel 2: Vote Distribution (Pie Chart)
    # ══════════════════════════════════════════════════════════
    ax2 = fig.add_subplot(gs[0, 2:4])

    wedges, texts, autotexts = ax2.pie(
        votes, labels=names, autopct='%1.1f%%',
        colors=[colors.get(n, '#888') for n in names],
        explode=[0.05 if n == winner else 0 for n in names],
        shadow=True, startangle=90
    )
    for autotext in autotexts:
        autotext.set_fontsize(12)
        autotext.set_fontweight('bold')
    ax2.set_title('Vote Distribution', fontsize=14, fontweight='bold')

    # ══════════════════════════════════════════════════════════
    # Panel 3-7: Trait Box Plots
    # ══════════════════════════════════════════════════════════
    traits = ['wisdom', 'fear', 'anger', 'openness', 'trust']
    trait_titles = ['🧠 Wisdom', '😨 Fear', '😠 Anger', '🌟 Openness', '🤝 Trust']

    for idx, (trait, title) in enumerate(zip(traits, trait_titles)):
        ax = fig.add_subplot(gs[1, idx]) if idx < 4 else fig.add_subplot(gs[2, 0])

        unique_votes = df["vote"].unique()
        data = [df[df["vote"] == name][trait].values for name in unique_votes]

        bp = ax.boxplot(data, tick_labels=unique_votes, patch_artist=True, widths=0.6)

        for patch, name in zip(bp['boxes'], unique_votes):
            patch.set_facecolor(colors.get(name, '#888'))
            patch.set_alpha(0.7)

        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_ylim(0, 11)
        ax.tick_params(axis='x', rotation=15, labelsize=9)
        ax.spines[['top', 'right']].set_visible(False)
        ax.axhline(y=5.5, color='gray', linestyle='--', alpha=0.3)

    # ══════════════════════════════════════════════════════════
    # Panel 8: Average Traits Comparison (Radar-like Bar)
    # ══════════════════════════════════════════════════════════
    ax8 = fig.add_subplot(gs[2, 1:3])

    trait_means = df.groupby('vote')[traits].mean()

    x = np.arange(len(traits))
    width = 0.35

    for i, (name, row) in enumerate(trait_means.iterrows()):
        offset = width * (i - 0.5)
        ax8.bar(x + offset, row.values, width, label=name,
               color=colors.get(name, '#888'), alpha=0.8)

    ax8.set_ylabel('Average Score', fontsize=11)
    ax8.set_title('📊 Average Traits by Vote Choice', fontsize=14, fontweight='bold')
    ax8.set_xticks(x)
    ax8.set_xticklabels([t.capitalize() for t in traits], fontsize=10)
    ax8.legend(loc='upper right')
    ax8.set_ylim(0, 10)
    ax8.spines[['top', 'right']].set_visible(False)
    ax8.axhline(y=5, color='gray', linestyle='--', alpha=0.3)

    # ══════════════════════════════════════════════════════════
    # Panel 9: Voter Wisdom-Fear Scatter
    # ══════════════════════════════════════════════════════════
    ax9 = fig.add_subplot(gs[2, 3])

    for name in df['vote'].unique():
        subset = df[df['vote'] == name]
        ax9.scatter(subset['wisdom'], subset['fear'],
                   c=colors.get(name, '#888'), label=name,
                   alpha=0.6, s=50, edgecolors='white')

    ax9.set_xlabel('Wisdom', fontsize=11)
    ax9.set_ylabel('Fear', fontsize=11)
    ax9.set_title('🎯 Wisdom vs Fear Distribution', fontsize=14, fontweight='bold')
    ax9.legend(loc='best', fontsize=9)
    ax9.set_xlim(0, 11)
    ax9.set_ylim(0, 11)
    ax9.spines[['top', 'right']].set_visible(False)
    ax9.axhline(y=5.5, color='gray', linestyle='--', alpha=0.3)
    ax9.axvline(x=5.5, color='gray', linestyle='--', alpha=0.3)

    # ══════════════════════════════════════════════════════════
    # Add Statistics Text Box
    # ══════════════════════════════════════════════════════════
    stats_text = (
        f"📈 Statistics Summary\n"
        f"{'─'*30}\n"
        f"Total Voters: {len(voters)}\n"
        f"Winner: {winner}\n"
        f"Margin: {max(votes) - min(votes)} votes ({(max(votes)-min(votes))/total*100:.1f}%)\n\n"
    )

    for name in names:
        subset = df[df['vote'] == name]
        stats_text += f"{name}:\n"
        stats_text += f"  Votes: {results.get(name, 0)}\n"
        stats_text += f"  Avg Wisdom: {subset['wisdom'].mean():.2f}\n"
        stats_text += f"  Avg Fear: {subset['fear'].mean():.2f}\n\n"

    # Add text box in empty space
    fig.text(0.02, 0.02, stats_text, fontsize=10, fontfamily='monospace',
             verticalalignment='bottom', bbox=dict(boxstyle='round',
             facecolor='wheat', alpha=0.5))

    # Save figure
    plt.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"\n✅ Comprehensive report saved: {save_path}")
    print(f"   Resolution: {dpi} DPI")
    print(f"   Size: {fig.get_size_inches()[0]}\" x {fig.get_size_inches()[1]}\"")

    return fig


def create_multi_scenario_report(all_results, candidates,
                                  save_path="multi_scenario_report.png", dpi=200):
    """
    Create a comprehensive report comparing all scenarios.
    """

    n_scenarios = len(all_results)

    # Prepare comparison data
    comparison_data = []
    for scenario_key, data in all_results.items():
        results = data["results"]
        total = sum(results.values())
        comparison_data.append({
            "Scenario": scenario_key,
            "Trump": results.get("Donald Trump", 0),
            "Trump_pct": results.get("Donald Trump", 0) / total * 100,
            "Harris": results.get("Kamala Harris", 0),
            "Harris_pct": results.get("Kamala Harris", 0) / total * 100,
            "Winner": max(results, key=results.get),
            "Margin": abs(results.get("Donald Trump", 0) - results.get("Kamala Harris", 0)) / total * 100
        })

    comp_df = pd.DataFrame(comparison_data)

    # Create figure
    fig = plt.figure(figsize=(20, 14))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.25)

    fig.suptitle('Multi-Scenario Election Simulation Report',
                 fontsize=20, fontweight='bold', y=0.98)

    colors = {"Donald Trump": "#E63946", "Kamala Harris": "#457B9D"}

    # ══════════════════════════════════════════════════════════
    # Panel 1: Vote Percentages Comparison
    # ══════════════════════════════════════════════════════════
    ax1 = fig.add_subplot(gs[0, 0:2])

    x = np.arange(n_scenarios)
    width = 0.35

    bars1 = ax1.bar(x - width/2, comp_df["Trump_pct"], width,
                    label='Donald Trump', color=colors["Donald Trump"])
    bars2 = ax1.bar(x + width/2, comp_df["Harris_pct"], width,
                    label='Kamala Harris', color=colors["Kamala Harris"])

    ax1.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax1.set_ylabel('Vote Percentage (%)', fontsize=12)
    ax1.set_title('📊 Vote Percentage by Scenario', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(comp_df["Scenario"], rotation=15, ha='right', fontsize=10)
    ax1.legend(loc='upper right')
    ax1.set_ylim(0, 100)
    ax1.spines[['top', 'right']].set_visible(False)

    # Add percentage labels
    for bar in bars1:
        ax1.annotate(f'{bar.get_height():.1f}%',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax1.annotate(f'{bar.get_height():.1f}%',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

    # ══════════════════════════════════════════════════════════
    # Panel 2: Win/Loss Summary
    # ══════════════════════════════════════════════════════════
    ax2 = fig.add_subplot(gs[0, 2])

    trump_wins = (comp_df["Winner"] == "Donald Trump").sum()
    harris_wins = (comp_df["Winner"] == "Kamala Harris").sum()

    wedges, texts, autotexts = ax2.pie(
        [trump_wins, harris_wins],
        labels=['Trump Wins', 'Harris Wins'],
        autopct=lambda pct: f'{int(pct/100*n_scenarios)} scenarios',
        colors=[colors["Donald Trump"], colors["Kamala Harris"]],
        explode=[0.05, 0.05],
        shadow=True,
        startangle=90
    )
    ax2.set_title('🏆 Scenario Wins', fontsize=14, fontweight='bold')

    # ══════════════════════════════════════════════════════════
    # Panel 3: Margin of Victory
    # ══════════════════════════════════════════════════════════
    ax3 = fig.add_subplot(gs[1, 0])

    margin_colors = [colors[w] for w in comp_df["Winner"]]
    bars = ax3.barh(comp_df["Scenario"], comp_df["Margin"], color=margin_colors, alpha=0.8)

    ax3.set_xlabel('Margin (%)', fontsize=11)
    ax3.set_title('📏 Victory Margin by Scenario', fontsize=14, fontweight='bold')
    ax3.spines[['top', 'right']].set_visible(False)

    for bar, winner in zip(bars, comp_df["Winner"]):
        ax3.annotate(f'{winner.split()[1]}',
                    xy=(bar.get_width(), bar.get_y() + bar.get_height()/2),
                    xytext=(3, 0), textcoords="offset points",
                    ha='left', va='center', fontsize=9)

    # ══════════════════════════════════════════════════════════
    # Panel 4: Average Traits per Scenario
    # ══════════════════════════════════════════════════════════
    ax4 = fig.add_subplot(gs[1, 1:3])

    # Calculate average traits for Trump voters in each scenario
    scenario_traits = []
    for scenario_key, data in all_results.items():
        voters = data["voters"]
        trump_voters = [v for v in voters if v.get("vote") == "Donald Trump"]
        harris_voters = [v for v in voters if v.get("vote") == "Kamala Harris"]

        if trump_voters:
            trump_wisdom = np.mean([v["traits"]["wisdom"] for v in trump_voters])
            trump_fear = np.mean([v["traits"]["fear"] for v in trump_voters])
        else:
            trump_wisdom, trump_fear = 0, 0

        if harris_voters:
            harris_wisdom = np.mean([v["traits"]["wisdom"] for v in harris_voters])
            harris_fear = np.mean([v["traits"]["fear"] for v in harris_voters])
        else:
            harris_wisdom, harris_fear = 0, 0

        scenario_traits.append({
            "Scenario": scenario_key,
            "Trump_Wisdom": trump_wisdom,
            "Trump_Fear": trump_fear,
            "Harris_Wisdom": harris_wisdom,
            "Harris_Fear": harris_fear
        })

    traits_df = pd.DataFrame(scenario_traits)

    x = np.arange(n_scenarios)
    width = 0.2

    ax4.bar(x - 1.5*width, traits_df["Trump_Wisdom"], width,
           label='Trump Voter Wisdom', color=colors["Donald Trump"], alpha=0.9)
    ax4.bar(x - 0.5*width, traits_df["Trump_Fear"], width,
           label='Trump Voter Fear', color=colors["Donald Trump"], alpha=0.5)
    ax4.bar(x + 0.5*width, traits_df["Harris_Wisdom"], width,
           label='Harris Voter Wisdom', color=colors["Kamala Harris"], alpha=0.9)
    ax4.bar(x + 1.5*width, traits_df["Harris_Fear"], width,
           label='Harris Voter Fear', color=colors["Kamala Harris"], alpha=0.5)

    ax4.set_ylabel('Average Trait Score', fontsize=11)
    ax4.set_title('🧠 Voter Traits by Scenario', fontsize=14, fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels(traits_df["Scenario"], rotation=15, ha='right', fontsize=10)
    ax4.legend(loc='upper right', fontsize=9)
    ax4.set_ylim(0, 10)
    ax4.spines[['top', 'right']].set_visible(False)

    # ══════════════════════════════════════════════════════════
    # Summary Statistics Text
    # ══════════════════════════════════════════════════════════
    summary_text = (
        f"📈 Summary Statistics\n"
        f"{'─'*35}\n"
        f"Total Scenarios: {n_scenarios}\n"
        f"Trump Wins: {trump_wins}\n"
        f"Harris Wins: {harris_wins}\n\n"
        f"Largest Trump Margin: {comp_df[comp_df['Winner']=='Donald Trump']['Margin'].max():.1f}%\n"
        f"Largest Harris Margin: {comp_df[comp_df['Winner']=='Kamala Harris']['Margin'].max():.1f}%\n\n"
        f"Average Trump %: {comp_df['Trump_pct'].mean():.1f}%\n"
        f"Average Harris %: {comp_df['Harris_pct'].mean():.1f}%"
    )

    fig.text(0.02, 0.02, summary_text, fontsize=10, fontfamily='monospace',
             verticalalignment='bottom', bbox=dict(boxstyle='round',
             facecolor='wheat', alpha=0.5))

    plt.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"\n✅ Multi-scenario report saved: {save_path}")

    return fig


# ══════════════════════════════════════════════════════════════
# GENERATE REPORTS
# ══════════════════════════════════════════════════════════════

# Check for loaded data or current simulation data
if 'loaded_voters' in dir() and loaded_voters:
    print("📊 Using loaded data for report...")
    create_comprehensive_report(
        loaded_voters,
        loaded_results,
        loaded_candidates,
        scenario_name="Loaded Data",
        save_path="comprehensive_report_loaded.png"
    )
elif 'voters' in dir() and voters:
    print("📊 Using current simulation data for report...")
    scenario_name = SELECTED_SCENARIO if 'SELECTED_SCENARIO' in dir() else ""
    create_comprehensive_report(
        voters,
        results,
        CANDIDATES,
        scenario_name=scenario_name,
        save_path="comprehensive_report.png"
    )
else:
    print("⚠️ No data available for report.")
    print("   Run simulation (Cell 9) or load data (Cell 12) first.")

# Multi-scenario report
if 'all_results' in dir() and all_results:
    print("\n📊 Creating multi-scenario comparison report...")
    create_multi_scenario_report(
        all_results,
        CANDIDATES,
        save_path="multi_scenario_report.png"
    )

print("\n✅ Report functions defined:")
print("   • create_comprehensive_report() - Single scenario report")
print("   • create_multi_scenario_report() - Multi-scenario comparison")

In [ ]:
#@title Cell 14: Download All Files (Colab)

from google.colab import files
import os
import zipfile

def download_all_files(directories=None, zip_name="simulation_results.zip"):
    """
    Create a zip file of all results and download it.
    """

    if directories is None:
        directories = ["simulation_data", "multi_scenario_data"]

    # Find all PNG files in current directory
    png_files = [f for f in os.listdir('.') if f.endswith('.png')]

    # Create zip file
    print(f"📦 Creating {zip_name}...")

    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add directories
        for directory in directories:
            if os.path.exists(directory):
                for root, dirs, files_list in os.walk(directory):
                    for file in files_list:
                        file_path = os.path.join(root, file)
                        arcname = file_path
                        zipf.write(file_path, arcname)
                        print(f"  ✅ Added: {arcname}")

        # Add PNG files
        for png_file in png_files:
            zipf.write(png_file)
            print(f"  ✅ Added: {png_file}")

    # Get zip size
    zip_size = os.path.getsize(zip_name) / 1024
    print(f"\n📦 Zip file created: {zip_name} ({zip_size:.1f} KB)")

    # Download
    print("⬇️ Starting download...")
    files.download(zip_name)

    return zip_name


def list_all_files():
    """List all saved files."""
    print("="*60)
    print(" 📁 SAVED FILES")
    print("="*60)

    # Current directory PNGs
    png_files = [f for f in os.listdir('.') if f.endswith('.png')]
    if png_files:
        print("\n📊 Report Images:")
        for f in png_files:
            size = os.path.getsize(f) / 1024
            print(f"   • {f} ({size:.1f} KB)")

    # Data directories
    for directory in ["simulation_data", "multi_scenario_data"]:
        if os.path.exists(directory):
            print(f"\n📁 {directory}/")
            for root, dirs, files_list in os.walk(directory):
                level = root.replace(directory, '').count(os.sep)
                indent = '   ' * (level + 1)
                subindent = '   ' * (level + 2)

                if root != directory:
                    print(f"{indent}📂 {os.path.basename(root)}/")

                for file in files_list:
                    file_path = os.path.join(root, file)
                    size = os.path.getsize(file_path) / 1024
                    print(f"{subindent}• {file} ({size:.1f} KB)")


# List files
list_all_files()

# Download button
print("\n" + "="*60)
print(" ⬇️ DOWNLOAD")
print("="*60)
print("\nRun the following to download all files:")
print(">>> download_all_files()")


```
📁 simulation_data/
   ├── candidates.json          # اطلاعات کاندیداها
   ├── debate_transcript.json   # متن کامل مناظره
   ├── voters.json              # اطلاعات رأی‌دهندگان
   ├── results.json             # نتایج انتخابات
   ├── metadata.json            # متادیتای شبیه‌سازی
   └── full_simulation.pkl      # بکاپ کامل (pickle)

📁 multi_scenario_data/
   ├── debate_transcript.json   # مناظره مشترک
   ├── candidates.json          # کاندیداها
   ├── all_scenarios_summary.json
   ├── full_multi_scenario.pkl
   ├── 📂 Scenario_1/
   │   ├── voters.json
   │   └── results.json
   ├── 📂 Scenario_2/
   │   └── ...
   └── ...

📊 Report Images:
   ├── comprehensive_report.png      # گزارش جامع تک سناریو
   └── multi_scenario_report.png     # گزارش مقایسه‌ای
```



In [ ]:
# ذخیره داده‌ها
save_all_data(CANDIDATES, transcript, context, voters, results, "Scenario 1")

# بارگذاری داده‌ها
data = load_all_data("simulation_data")
voters = data['voters']
results = data['results']

# تولید گزارش
create_comprehensive_report(voters, results, candidates)

# دانلود همه فایل‌ها
download_all_files()